### Persona-Conditioned Abstractive Summarization Engine

This notebook implements the **Controlled Text Generation & Abstractive Summarization** module for the University Communications Intelligence pipeline.

It fine-tunes a sequence-to-sequence transformer model (`facebook/bart-base`) to transform lengthy institutional circulars into concise, role-tailored summaries for specific target audiences (**Students**, **Faculty**, and **Administrators**).

---

### Key Features & Pipeline Capabilities
* Task-Conditioned Generation: Utilizes role-prefixed prompts (e.g., `"summarize for student: ..."` vs. `"summarize for faculty: ..."`) to control summary scope and tone.
* Hugging Face Trainer Integration: Leverages `Seq2SeqTrainer` and `DataCollatorForSeq2Seq` for efficient fine-tuning and dynamic batch padding.
* Evaluation & Guardrail Framework:
  * ROUGE Metrics: Computes `ROUGE-1`, `ROUGE-2`, and `ROUGE-L` scores on held-out evaluation splits to measure summary quality.
  * Entity Groundedness Guardrail: Uses `spaCy` entity matching to detect and alert on date/time hallucinations missing from source documents.


In [5]:
import json
import pandas as pd
from datasets import Dataset
from pathlib import Path

DATA_PATH = r"data\summarizerdata\training_pairs.jsonl"

records = []

relative_path = Path(DATA_PATH.replace("\\", "/"))

if relative_path.is_absolute():
    data_file = relative_path
else:
    candidates = [base / relative_path for base in [Path.cwd(), *Path.cwd().parents]]
    data_file = next((path for path in candidates if path.is_file()), None)

if data_file is None:
    raise FileNotFoundError(
        f"Could not find {DATA_PATH!r}. Current working directory: {Path.cwd()}"
    )

with data_file.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} notices total")


formatted_rows = []
skipped_no_text = 0
for item in records:
    raw_notice = (item.get("cleaned_text") or item.get("raw_text") or "").strip()
    if not raw_notice:
        skipped_no_text += 1
        continue

    student_summary = item.get("student_summary")
    if student_summary:
        formatted_rows.append({
            "input_text": f"summarize for student: {raw_notice}",
            "target_summary": student_summary,
        })

    faculty_summary = item.get("faculty_summary")
    if faculty_summary:
        formatted_rows.append({
            "input_text": f"summarize for faculty: {raw_notice}",
            "target_summary": faculty_summary,
        })

print(f"Skipped {skipped_no_text} notices with no extracted text")
print(f"Built {len(formatted_rows)} (input, target) training rows from labeled summaries")

if len(formatted_rows) < 20:
    print(
        "WARNING: this is a very small number of labeled examples for "
        "fine-tuning a seq2seq model. Expect the model to memorize/overfit "
        "rather than generalize -- see the note at the end of the notebook."
    )

raw_dataset_full = Dataset.from_pandas(pd.DataFrame(formatted_rows))
split = raw_dataset_full.train_test_split(test_size=0.2, seed=42)
raw_dataset = split["train"]
eval_dataset = split["test"]

print("\nSample Dataset Row:")
# The JSONL already contains input_text and target_summary fields.
formatted_rows = [
    {
        "input_text": item["input_text"],
        "target_summary": item["target_summary"],
    }
    for item in records
    if item.get("input_text") and item.get("target_summary")
]

raw_dataset_full = Dataset.from_list(formatted_rows)
split = raw_dataset_full.train_test_split(test_size=0.2, seed=42)
raw_dataset = split["train"]
eval_dataset = split["test"]

print(raw_dataset[0])
print(f"\nTrain rows: {len(raw_dataset)} | Eval rows: {len(eval_dataset)}")

Loaded 150 notices total
Skipped 150 notices with no extracted text
Built 0 (input, target) training rows from labeled summaries

Sample Dataset Row:
{'input_text': 'summarize for student: Master of Arts in Social Work\nVenue- Department of Sociology, Faculty of Social Sciences,\nBanaras Hindu University, Varanasi UP, PIN 221005\nSl.No. Application Number GD-PI _ Date Time\n1 265710232524 June 24, 2026 10:00 AM\n2 265710008885 June 24, 2026 10:00 AM\n3 265710011386 June 24, 2026 10:00 AM\n4 265710428728 June 24, 2026 10:00 AM\n5 265710154987 June 24, 2026 10:00 AM\n6 265710058495 June 24, 2026 10:00 AM\n7 265710286165 June 24, 2026 10:00 AM\n8 265710020480 June 24, 2026 10:00 AM\n9 265710117002 June 24, 2026 10:00 AM\n10 265710089104 June 24, 2026 10:00 AM\n11 265710380819 June 24, 2026 10:00 AM\n12 265710157154 June 24, 2026 10:00 AM\n13 265710115216 June 24, 2026 10:00 AM\n14 265710181233 June 24, 2026 10:00 AM\n15 265710196773 June 24, 2026 10:00 AM\n16 265710314186 June 24, 2026 10

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

model_name = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def preprocess_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(text_target=examples["target_summary"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = raw_dataset.map(preprocess_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(preprocess_function, batched=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="./results_bart",
    num_train_epochs=3,             
    per_device_train_batch_size=2,
    eval_strategy="epoch",
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_eval_dataset,
    processing_class=tokenizer,
)

print("Starting Fine-Tuning Loop...")
trainer.train()
print("Training Complete!")

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Starting Fine-Tuning Loop...


Epoch,Training Loss,Validation Loss
1,1.739771,1.438713
2,1.269144,0.676202
3,0.388428,0.622742


Training Complete!


In [8]:
def generate_persona_summary(raw_notice: str, role: str = "student") -> str:
    input_prompt = f"summarize for {role}: {raw_notice}"
    inputs = tokenizer(input_prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    
    summary_ids = model.generate(
        inputs["input_ids"],
        max_new_tokens=60,
        min_length=10,
        num_beams=4,
        no_repeat_ngram_size=3,        
        repetition_penalty=1.2,        
        early_stopping=True
    )
    
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Test User Inputs
test_notice = "CIRCULAR: Library books must be returned by September 30. Faculty approvals needed for extensions."

print("\n--- USER GENERATION TEST ---")
print("Student View:", generate_persona_summary(test_notice, role="student"))
print("Faculty View:", generate_persona_summary(test_notice, role="faculty"))


--- USER GENERATION TEST ---
Student View: Find your application number on this CIRCULAR: Library books must be returned by September 30. Faculty approvals needed for extensions.
Faculty View: The department running CIRCULAR admissions should be ready to conduct the listed GD-PI/practical tests at the notified venue around September 30.


In [9]:
import evaluate
import spacy

# 1. ROUGE Metrics Evaluation
rouge = evaluate.load("rouge")

# Evaluate on the held-out eval split (not the training data) so ROUGE reflects
# generalization rather than memorization.
def split_role_and_text(input_text):
    role, _, notice = input_text.partition(": ")
    role = role.replace("summarize for ", "")
    return role, notice

predictions = []
references = []
for row in eval_dataset:
    role, notice = split_role_and_text(row["input_text"])
    predictions.append(generate_persona_summary(notice, role=role))
    references.append(row["target_summary"])

rouge_results = rouge.compute(predictions=predictions, references=references)
print("\n--- EVALUATION RESULTS ---")
print("ROUGE Metrics:", rouge_results)

# 2. Entity Hallucination Check
nlp = spacy.load("en_core_web_sm")

def check_date_hallucinations(source_text: str, generated_summary: str) -> bool:
    source_doc = nlp(source_text)
    summary_doc = nlp(generated_summary)
    
    source_dates = {ent.text.lower() for ent in source_doc.ents if ent.label_ in ["DATE", "TIME"]}
    summary_dates = {ent.text.lower() for ent in summary_doc.ents if ent.label_ in ["DATE", "TIME"]}
    
    # Check if generated summary contains dates missing from the source text
    hallucinated_dates = summary_dates - source_dates
    has_hallucination = len(hallucinated_dates) > 0
    
    if has_hallucination:
        print(f"Hallucination Warning! Generated ungrounded dates: {hallucinated_dates}")
    else:
        print("Date Check Passed: Zero date hallucinations detected.")
        
    return has_hallucination

# Test Guardrail Check
check_date_hallucinations(test_notice, generate_persona_summary(test_notice, role="student"))


--- EVALUATION RESULTS ---
ROUGE Metrics: {'rouge1': np.float64(0.4172630557901089), 'rouge2': np.float64(0.28856318979848544), 'rougeL': np.float64(0.3565578753668961), 'rougeLsum': np.float64(0.35225701717411706)}
Date Check Passed: Zero date hallucinations detected.


False